In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
import os
BASE = "/content/drive/MyDrive/Nigehbaan dataset(merged)"
MERGED_PATH = os.path.join(BASE, "nigehbaan_merged_dataset.csv")
WINDOW_SAMPLES = 200
STRIDE = 50
CHANNELS = ['acc_x','acc_y','acc_z','gyro_x','gyro_y','gyro_z']
BATCH_SIZE = 128
NUM_CLASSES = 3
CLASS_NAMES = ['crash','normal','snatch']
OUTPUT_DIR = os.path.join(BASE, "nigehbaan_models")
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("Merged path:", MERGED_PATH)


Merged path: /content/drive/MyDrive/Nigehbaan dataset(merged)/nigehbaan_merged_dataset.csv


In [ ]:
import pandas as pd

index_path = os.path.join(BASE, "window_index.csv")

if not os.path.exists(index_path):
    print("Building window index (streaming) ...")

    total_rows = 0
    !wc -l "{MERGED_PATH}"

    CHUNK = 200000
    reader = pd.read_csv(MERGED_PATH, chunksize=CHUNK)
    window_records = []
    cumulative_rows = 0

    from collections import deque
    buf = pd.DataFrame(columns=CHANNELS + ['label'])
    for chunk in reader:
        chunk = chunk[CHANNELS + ['label']]
        buf = pd.concat([buf, chunk], ignore_index=True)
        while len(buf) >= WINDOW_SAMPLES:
            window = buf.iloc[:WINDOW_SAMPLES]
            label = window['label'].mode().iloc[0]
            window_records.append((cumulative_rows, label))
            buf = buf.iloc[STRIDE:].reset_index(drop=True)
            cumulative_rows += STRIDE

        pass


    df = pd.read_csv(MERGED_PATH)
    N = len(df)
    idxs = []
    labels = []
    for start in range(0, N - WINDOW_SAMPLES + 1, STRIDE):
        window_labels = df['label'].iloc[start:start+WINDOW_SAMPLES]
        label = window_labels.mode().iloc[0]
        idxs.append(start)
        labels.append(label)
    idx_df = pd.DataFrame({"start": idxs, "label": labels})
    idx_df.to_csv(index_path, index=False)
    print("Window index built:", index_path, "windows:", len(idxs))
else:
    print("Index already exists:", index_path)


Index already exists: /content/drive/MyDrive/Nigehbaan dataset(merged)/window_index.csv


In [ ]:
import numpy as np
import pandas as pd
import random
from tensorflow.keras.utils import Sequence

class WindowGenerator(Sequence):
    def __init__(self, merged_csv, index_csv, channels, batch_size=128, window=200, shuffle=True, classes=None):
        self.merged_csv = merged_csv
        self.index_df = pd.read_csv(index_csv)
        self.channels = channels
        self.batch_size = batch_size
        self.window = window
        self.shuffle = shuffle
        self.classes = classes or ['crash','normal','snatch']
        self.on_epoch_end()

    def __len__(self):
        return int(np.ceil(len(self.index_df) / self.batch_size))

    def on_epoch_end(self):
        self.indices = np.arange(len(self.index_df))
        if self.shuffle:
            np.random.shuffle(self.indices)

    def __getitem__(self, idx):
        batch_indices = self.indices[idx*self.batch_size:(idx+1)*self.batch_size]
        X_batch = np.zeros((len(batch_indices), self.window, len(self.channels)), dtype=np.float32)
        y_batch = np.zeros((len(batch_indices),), dtype=np.int32)


        df = pd.read_csv(self.merged_csv)
        for i, bi in enumerate(batch_indices):
            start = int(self.index_df['start'].iloc[bi])
            window_df = df.iloc[start:start+self.window]
            X_batch[i] = window_df[self.channels].values
            lab = self.index_df['label'].iloc[bi]
            y_batch[i] = self.classes.index(lab)
        return X_batch, np.eye(len(self.classes))[y_batch]

index_csv = index_path
gen = WindowGenerator(MERGED_PATH, index_csv, CHANNELS, batch_size=BATCH_SIZE, window=WINDOW_SAMPLES, shuffle=True, classes=CLASS_NAMES)
X, y = gen[0]
print("Batch X shape:", X.shape, "Batch y shape:", y.shape)


Batch X shape: (128, 200, 6) Batch y shape: (128, 3)


In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models

def build_cnn_lstm(window_len=WINDOW_SAMPLES, n_channels=len(CHANNELS), n_classes=NUM_CLASSES):
    inp = layers.Input(shape=(window_len, n_channels))
    x = layers.Conv1D(64, kernel_size=5, activation='relu', padding='same')(inp)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPool1D(pool_size=2)(x)

    x = layers.Conv1D(128, kernel_size=3, activation='relu', padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPool1D(pool_size=2)(x)

    x = layers.Dropout(0.25)(x)
    x = layers.LSTM(64, return_sequences=False)(x)

    x = layers.Dense(64, activation='relu')(x)
    x = layers.Dropout(0.3)(x)
    out = layers.Dense(n_classes, activation='softmax')(x)

    model = models.Model(inputs=inp, outputs=out)
    model.compile(optimizer='adam',
                  loss='categorical_crossentropy',
                  metrics=['accuracy'])
    return model

model = build_cnn_lstm()
model.summary()


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 200, 6)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d (Conv1D)                 │ (None, 200, 64)        │         1,984 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 200, 64)        │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d (MaxPooling1D)    │ (None, 100, 64)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_1 (Conv1D)               │ (None, 100, 128)       │        24,704 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 100, 128)       │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_1 (MaxPooling1D)  │ (None, 50, 128)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 50, 128)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ (None, 64)             │        49,408 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 64)             │         4,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 3)              │           195 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 81,219 (317.26 KB)

 Trainable params: 80,835 (315.76 KB)

 Non-trainable params: 384 (1.50 KB)

In [ ]:
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau
import os
import pandas as pd


index_path = os.path.join(BASE, "window_index.csv")
idx_df = pd.read_csv(index_path)


from sklearn.model_selection import train_test_split

train_idx, val_idx = train_test_split(
    idx_df,
    test_size=0.2,
    stratify=idx_df['label'],
    random_state=42
)

train_idx.to_csv(os.path.join(BASE, "train_index.csv"), index=False)
val_idx.to_csv(os.path.join(BASE, "val_index.csv"), index=False)

print("Train windows:", len(train_idx))
print("Val windows:", len(val_idx))

train_gen = WindowGenerator(
    MERGED_PATH,
    os.path.join(BASE, "train_index.csv"),
    CHANNELS,
    batch_size=BATCH_SIZE,
    window=WINDOW_SAMPLES,
    shuffle=True,
    classes=CLASS_NAMES
)

val_gen = WindowGenerator(
    MERGED_PATH,
    os.path.join(BASE, "val_index.csv"),
    CHANNELS,
    batch_size=BATCH_SIZE,
    window=WINDOW_SAMPLES,
    shuffle=False,
    classes=CLASS_NAMES
)


ckpt_path = os.path.join(OUTPUT_DIR, "best_cnn_lstm.h5")

callbacks = [
    ModelCheckpoint(
        ckpt_path,
        save_best_only=True,
        monitor='val_accuracy',
        mode='max'
    ),
    EarlyStopping(
        monitor='val_accuracy',
        patience=8,
        restore_best_weights=True
    ),
    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=3,
        verbose=1
    )
]


history = model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=30,
    callbacks=callbacks
)


Train windows: 25287
Val windows: 6322


/usr/local/lib/python3.12/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/30
198/198 ━━━━━━━━━━━━━━━━━━━━ 0s 3s/step - accuracy: 0.8471 - loss: nan

/usr/local/lib/python3.12/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


198/198 ━━━━━━━━━━━━━━━━━━━━ 711s 4s/step - accuracy: 0.8471 - loss: nan - val_accuracy: 0.8567 - val_loss: nan - learning_rate: 0.0010
Epoch 2/30
198/198 ━━━━━━━━━━━━━━━━━━━━ 710s 4s/step - accuracy: 0.8577 - loss: nan - val_accuracy: 0.8567 - val_loss: nan - learning_rate: 0.0010
Epoch 3/30
198/198 ━━━━━━━━━━━━━━━━━━━━ 0s 3s/step - accuracy: 0.8589 - loss: nan
Epoch 3: ReduceLROnPlateau reducing learning rate to 0.0005000000237487257.
198/198 ━━━━━━━━━━━━━━━━━━━━ 723s 4s/step - accuracy: 0.8589 - loss: nan - val_accuracy: 0.8567 - val_loss: nan - learning_rate: 0.0010
Epoch 4/30
198/198 ━━━━━━━━━━━━━━━━━━━━ 717s 4s/step - accuracy: 0.8561 - loss: nan - val_accuracy: 0.8567 - val_loss: nan - learning_rate: 5.0000e-04
Epoch 5/30
198/198 ━━━━━━━━━━━━━━━━━━━━ 707s 4s/step - accuracy: 0.8598 - loss: nan - val_accuracy: 0.8567 - val_loss: nan - learning_rate: 5.0000e-04
Epoch 6/30
198/198 ━━━━━━━━━━━━━━━━━━━━ 0s 3s/step - accuracy: 0.8593 - loss: nan
Epoch 6: ReduceLROnPlateau reducing lea

In [ ]:
import numpy as np
import pandas as pd
from tensorflow.keras.utils import Sequence

class WindowGenerator(Sequence):
    def __init__(self, merged_csv, index_csv, channels, batch_size=128,
                 window=200, shuffle=True, classes=None):

        self.df = pd.read_csv(merged_csv)[channels + ['label']].reset_index(drop=True)

        self.index_df = pd.read_csv(index_csv)

        self.channels = channels
        self.batch_size = batch_size
        self.window = window
        self.shuffle = shuffle
        self.classes = classes or ['crash', 'normal', 'snatch']

        self.class_to_index = {c: i for i in self.classes}

        self.on_epoch_end()

    def __len__(self):
        return int(np.ceil(len(self.index_df) / self.batch_size))

    def on_epoch_end(self):
        """Shuffle window order after each epoch"""
        self.indices = np.arange(len(self.index_df))
        if self.shuffle:
            np.random.shuffle(self.indices)

    def __getitem__(self, idx):
        """Return (batch_X, batch_y)"""
        batch_indices = self.indices[idx * self.batch_size:(idx + 1) * self.batch_size]
        batch_size_actual = len(batch_indices)

        X_batch = np.zeros((batch_size_actual, self.window, len(self.channels)), dtype=np.float32)
        y_batch = np.zeros((batch_size_actual,), dtype=np.int32)

        for i, bi in enumerate(batch_indices):
            start_pos = int(self.index_df.iloc[bi]['start'])

            window_data = self.df.iloc[start_pos:start_pos + self.window]

            X_batch[i] = window_data[self.channels].values

            label_str = self.index_df.iloc[bi]['label']
            y_batch[i] = self.class_to_index[label_str]

        y_onehot = np.eye(len(self.classes))[y_batch]

        return X_batch, y_onehot


In [ ]:
model.load_weights(ckpt_path)

val_loss, val_acc = model.evaluate(val_gen)
print("Validation Accuracy:", val_acc)
print("Validation Loss:", val_loss)


50/50 ━━━━━━━━━━━━━━━━━━━━ 138s 3s/step - accuracy: 0.8587 - loss: nan
Validation Accuracy: 0.8566909432411194
Validation Loss: nan


In [ ]:
model_save_path = os.path.join(OUTPUT_DIR, "nigehbaan_cnn_lstm.keras")
model.save(model_save_path)
print("Saved model to:", model_save_path)


Saved model to: /content/drive/MyDrive/Nigehbaan dataset(merged)/nigehbaan_models/nigehbaan_cnn_lstm.keras


In [ ]:
import tensorflow as tf
import os

tflite_model_path = os.path.join(OUTPUT_DIR, "nigehbaan_cnn_lstm.tflite")

converter = tf.lite.TFLiteConverter.from_keras_model(model)

converter.target_spec.supported_ops = [
    tf.lite.OpsSet.TFLITE_BUILTINS,
    tf.lite.OpsSet.SELECT_TF_OPS
]

converter._experimental_lower_tensor_list_ops = False

converter.optimizations = [tf.lite.Optimize.DEFAULT]

tflite_model = converter.convert()

with open(tflite_model_path, "wb") as f:
    f.write(tflite_model)

print("TFLite model saved at:", tflite_model_path)



Saved artifact at '/tmp/tmpo69hfeeb'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 200, 6), dtype=tf.float32, name='keras_tensor')
Output Type:
  TensorSpec(shape=(None, 3), dtype=tf.float32, name=None)
Captures:
  135573556937360: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135573556938128: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135573556939856: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135573556937744: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135573556937936: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135573556940816: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135573556940240: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135573556939472: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135573556941776: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135573556941968: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135573556939088: Te

In [ ]:
tflite_path = "/content/drive/MyDrive/Nigehbaan dataset(merged)/nigehbaan_models/nigehbaan_cnn_lstm.tflite"
print("Found:", tflite_path)


Found: /content/drive/MyDrive/Nigehbaan dataset(merged)/nigehbaan_models/nigehbaan_cnn_lstm.tflite


In [ ]:
!pip install -q "tensorflow==2.12.0"
import tensorflow as tf
print("TF version:", tf.__version__)


ERROR: Could not find a version that satisfies the requirement tensorflow==2.12.0 (from versions: 2.16.0rc0, 2.16.1, 2.16.2, 2.17.0rc0, 2.17.0rc1, 2.17.0, 2.17.1, 2.18.0rc0, 2.18.0rc1, 2.18.0rc2, 2.18.0, 2.18.1, 2.19.0rc0, 2.19.0, 2.19.1, 2.20.0rc0, 2.20.0)
ERROR: No matching distribution found for tensorflow==2.12.0
TF version: 2.19.0
